# Ejercicio 3 - Seleccion y construccion de variables predictoras

Parte 2 del Laboratorio 4 (Machine Learning). Este cuaderno cubre el
ejercicio 3 completo: seleccion del conjunto de predictores (3.1),
diccionario de variables (3.2) e ingenieria de caracteristicas (3.3).

Depende de `data/processed/ml/dataset_ml.parquet` (ejercicio 1,
`notebooks/09_dataset_ml.ipynb`) y de la variable respuesta `cyano_alta`
(ejercicio 2, `notebooks/10_variable_respuesta.ipynb`). No reabre bandas
crudas; solo usa el contorno real del lago (ya calculado en la Parte I)
para las distancias geograficas.


## 0. Verificacion del trabajo previo (gate)

Antes de construir los predictores se exige que la variable respuesta del
ejercicio 2 pase su propio contrato. Si esto falla, el problema esta en
el ejercicio 2, no aqui.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import display

from src.config import LAGOS, VARIABLES_EXCLUIDAS_RESPUESTA
from src.respuesta import construir_respuesta, verificar_respuesta

resumen_gate_previo = verificar_respuesta()
print(f"Gate del ejercicio 2 correcto: {resumen_gate_previo['observaciones']} observaciones, "
      f"{resumen_gate_previo['positivos']} con cyano_alta=1.")

tabla_resp = construir_respuesta()


Gate del ejercicio 2 correcto: 492677 observaciones, 6365 con cyano_alta=1.


## 1. Ejercicio 3.1 - Seleccion del conjunto de predictores

El punto de partida es el dataset del ejercicio 1 mas la columna
`cyano_alta` del ejercicio 2. De ahi se excluyen explicitamente las
columnas identificadoras (`lago`, `fecha`, `x_utm`/`y_utm` se conservan
como predictoras espaciales, pero `lon`/`lat` no se usan por ser
redundantes con esas mismas coordenadas) y las variables que producirian
fuga de informacion hacia la variable respuesta.

La fuga es directa en un caso: `cianobacteria_ugl` es la variable de la
que se deriva `cyano_alta`, asi que nunca puede ser predictora. Es
indirecta en los otros dos: `B04` entra directamente en el NDCI que
calcula la clorofila-a de CyanoLakes (Parte I, ejercicio 3), y `ndvi` se
calcula como `(B08-B04)/(B08+B04)`, con lo que hereda esa misma fuga a
traves de B04. Esta lista vive en `config.VARIABLES_EXCLUIDAS_RESPUESTA`
y la usan tanto el ejercicio 2 como este; `verificar_anti_fuga` la vuelve
a chequear cada vez que se construye la matriz final, para que una fuga
nunca pase en silencio.


In [2]:
display(pd.Series(VARIABLES_EXCLUIDAS_RESPUESTA, name='razon').to_frame())

from src.features import FeaturesError, verificar_anti_fuga

try:
    verificar_anti_fuga(['B03', 'B08', 'ndwi', 'B04'])
except FeaturesError as error:
    print(f"El assert anti-fuga dispara correctamente si se inyecta una variable prohibida:\n{error}")


,razon
cianobacteria_ugl,es la variable de la que se deriva cyano_alta
B04,entra directamente en el NDCI que calcula la c...
ndvi,"se calcula como (B08-B04)/(B08+B04): usa B04, ..."


El assert anti-fuga dispara correctamente si se inyecta una variable prohibida:
Variables prohibidas por fuga se colaron en la matriz de predictores: ['B04']


Con esa lista descartada, el conjunto de predictores que llega al
ejercicio 3.3 combina:

- las bandas y el indice que sobreviven a la exclusion (`B03`, `B08`,
  `ndwi`),
- la posicion de la celda (`x_utm`, `y_utm`),
- variables temporales derivadas de la fecha (`mes`, `dia_anio_sin`,
  `dia_anio_cos`, `estacion`),
- una senal de calidad de la propia celda (`frac_valida`),
- tres variables derivadas que construye este cuaderno
  (`ratio_B03_B08`, `dist_orilla_m`, `dist_centroide_m`,
  `ndwi_vecindad_3x3`),
- y el one-hot de `lago` y `estacion`.

**Limitacion de la parte espectral:** el ejercicio 1 solo descargo tres
bandas (B03, B04, B08) mas la mascara de escena SCL (Parte I, ejercicio
2). B04 queda excluido por fuga, asi que la ingenieria de caracteristicas
solo puede combinar B03 y B08 entre si: no hay banda de borde rojo (B05,
la que usa el NDCI para el proxy de clorofila) ni infrarrojo de onda
corta disponibles para construir indices espectrales adicionales. Esto
acota el espacio de variables espectrales posibles a lo que ya se ve en
`ratio_B03_B08` y `ndwi`; un trabajo futuro que quisiera indices mas
especificos de cianobacteria (por ejemplo alguno basado en borde rojo)
tendria que volver a descargar bandas que este laboratorio no incluyo.


## 2. Ejercicio 3.2 - Diccionario de predictores

Cada variable que entra a la matriz final se documenta con su tipo, que
representa, por que se espera que aporte a distinguir `cyano_alta`, y su
fuente. El diccionario se completa dinamicamente para las columnas
one-hot de `lago` y `estacion` (no se pueden enumerar a mano porque
dependen de que lagos/estaciones aparezcan en los datos), y se escribe
junto con la matriz final a `results/tables/diccionario_predictores.csv`.


In [3]:
from src.features import DICCIONARIO_PREDICTORES

display(pd.DataFrame(DICCIONARIO_PREDICTORES).T)


,tipo,que_representa,por_que_contribuye,fuente
B03,banda espectral,Reflectancia de superficie en la banda verde (...,El agua con mas material en suspension/algas r...,Sentinel-2 L2A
B08,banda espectral,Reflectancia de superficie en el infrarrojo ce...,El NIR es muy sensible a materia organica/alga...,Sentinel-2 L2A
ndwi,indice,(B03-B08)/(B03+B08); cuanto de agua limpia hay...,Cianobacteria alta suele bajar el NDWI porque ...,Calculado en el ejercicio 3 de la Parte I
x_utm,caracteristica espacial,"Coordenada este del centroide de la celda, EPS...",Permite al modelo capturar patrones espaciales...,Calculado en el ejercicio 1
y_utm,caracteristica espacial,"Coordenada norte del centroide de la celda, EP...",Igual que x_utm: posicion absoluta dentro del ...,Calculado en el ejercicio 1
mes,caracteristica temporal,Mes calendario (1-12) de la fecha de la escena,Aproxima variacion estacional de temperatura y...,Derivado de la fecha oficial
dia_anio_sin,caracteristica temporal,Componente seno de la codificacion ciclica del...,Evita el salto artificial que tendria el dia d...,Derivado de la fecha oficial
dia_anio_cos,caracteristica temporal,Componente coseno de la codificacion ciclica d...,"Junto con dia_anio_sin, da al modelo una nocio...",Derivado de la fecha oficial
frac_valida,caracteristica de calidad,Fraccion de pixeles de 10 m validos dentro de ...,Una celda con menos pixeles validos promedia u...,Calculado en el ejercicio 1
ratio_B03_B08,derivada,B03/B08: contraste verde/infrarrojo cercano,Sensible a material particulado y biomasa en s...,"Ingenieria de caracteristicas, ejercicio 3.3"


## 3. Ejercicio 3.3 - Ingenieria de caracteristicas

Se documenta aqui, paso a paso, cada variable derivada que agrega este
ejercicio. Ninguna reabre un raster: todas se calculan a partir de las
columnas que ya trae el dataset del ejercicio 1 (mas el contorno real del
lago, ya calculado en la Parte I).


### 3.3.a Variables temporales

`mes` y `estacion` reutilizan la misma logica de estacion seca/lluviosa
de la Parte I (`comparacion_lagos.assign_season`). `dia_anio_sin` y
`dia_anio_cos` codifican el dia del anio de forma ciclica
(`sin(2*pi*dia/365.25)`, `cos(2*pi*dia/365.25)`) para que el modelo vea
el 31 de diciembre cerca del 1 de enero, en vez del salto artificial que
tendria el dia del anio como numero entero.


In [4]:
from src.features import agregar_temporales

enriquecida = agregar_temporales(tabla_resp)
display(
    enriquecida[['fecha', 'mes', 'estacion', 'dia_anio_sin', 'dia_anio_cos']]
    .drop_duplicates('fecha')
    .sort_values('fecha')
    .reset_index(drop=True)
)


,fecha,mes,estacion,dia_anio_sin,dia_anio_cos
0,2025-01-18,1,seca,0.304719,0.952442
1,2025-01-28,1,seca,0.463258,0.886223
2,2025-04-13,4,seca,0.979857,-0.199702
3,2025-04-15,4,seca,0.972408,-0.233289
4,2025-04-28,4,seca,0.896456,-0.443132
5,2025-05-13,5,lluviosa,0.753698,-0.657221
6,2025-07-17,7,lluviosa,-0.261414,-0.965227
7,2025-11-21,11,seca,-0.638384,0.769718
8,2025-11-24,11,seca,-0.597829,0.801624
9,2025-12-29,12,seca,-0.038696,0.999251


### 3.3.b Ratio de bandas

`ratio_B03_B08 = B03 / B08` es un contraste verde/infrarrojo cercano
distinto de cualquiera de las dos bandas por separado: sube cuando la
reflectancia verde domina sobre el infrarrojo, un patron asociado a mas
material particulado y biomasa en superficie. Cuando `B08` promedia
exactamente 0 en una celda (agua muy profunda y clara, sobre todo en
Atitlan) el ratio queda indefinido; no se inventa un valor para esos
casos, se documentan y se descartan mas adelante, en la construccion de
la matriz final.


In [5]:
from src.features import agregar_ratio_bandas

enriquecida = agregar_ratio_bandas(enriquecida)
display(enriquecida[['B03', 'B08', 'ratio_B03_B08']].describe())

indefinidos = enriquecida['ratio_B03_B08'].isna()
print(f"Celdas con ratio_B03_B08 indefinido (B08=0): {int(indefinidos.sum())} de {len(enriquecida)}")
display(enriquecida.loc[indefinidos, ['lago', 'fecha', 'B03', 'B08']].drop_duplicates())


,B03,B08,ratio_B03_B08
count,492677.000000,492677.000000,492663.000000
mean,0.023998,0.011588,3.392548
std,0.015034,0.010998,7.950475
min,0.005915,0.000000,0.238054
25%,0.015696,0.004296,1.806939
50%,0.021022,0.008558,2.400986
75%,0.026760,0.014976,3.337423
max,0.123480,0.171004,2672.000000


Celdas con ratio_B03_B08 indefinido (B08=0): 14 de 492677


,lago,fecha,B03,B08
61263,atitlan,2025-01-18,0.007750,0.0
230028,atitlan,2025-11-21,0.010986,0.0
230452,atitlan,2025-11-21,0.013388,0.0
230453,atitlan,2025-11-21,0.013865,0.0
230532,atitlan,2025-11-21,0.014423,0.0
230649,atitlan,2025-11-21,0.013295,0.0
230652,atitlan,2025-11-21,0.014105,0.0
230675,atitlan,2025-11-21,0.013954,0.0
230676,atitlan,2025-11-21,0.013507,0.0
230702,atitlan,2025-11-21,0.013905,0.0


**Lectura:** `ratio_B03_B08` tiene una mediana de 2.40 (rango
intercuartil 1.81-3.34): B03 casi siempre supera a B08 en agua, como se
espera de la reflectancia tipica del agua. Las 14 celdas indefinidas son
todas de Atitlan (13 del 2025-11-21 y 1 del 2025-01-18), agua profunda y
clara donde el infrarrojo cercano cae al ruido del sensor y promedia
exactamente 0 en la celda de 50 m; es la misma inestabilidad numerica de
Atitlan que ya aparecio en la Parte I. Se descartan esas 14 filas en vez
de imputar un valor, siguiendo el mismo criterio del resto del
laboratorio.


### 3.3.c Distancias geograficas

`dist_orilla_m` y `dist_centroide_m` son la distancia, en metros, de
cada celda al borde y al centroide del contorno real del lago (el mismo
poligono de OpenStreetMap que ya se uso en la Parte I, reproyectado a
EPSG:32615). Se espera que la cercania a la orilla aporte informacion
porque las floraciones tienden a acumularse en orillas y bahias, donde
hay menos mezcla del agua y mas nutrientes de origen terrestre cerca.


In [6]:
from src.features import agregar_distancias_geograficas

enriquecida = agregar_distancias_geograficas(enriquecida)
display(enriquecida.groupby('lago')[['dist_orilla_m', 'dist_centroide_m']].describe().T)


lago                       amatitlan        atitlan
dist_orilla_m    count  60642.000000  432035.000000
                 mean     280.162140    1323.224609
                 std      206.259598     948.136414
                 min        0.134598       1.409833
                 25%      116.585907     506.995880
                 50%      234.841660    1156.074097
                 75%      395.952118    2007.951904
                 max     1034.776733    3679.097168
dist_centroide_m count  60642.000000  432035.000000
                 mean    2963.089600    4792.997559
                 std     1279.817749    2133.453125
                 min        7.829033      21.002047
                 25%     2093.633362    3205.606445
                 50%     3055.666016    4736.397949
                 75%     3883.511475    6256.652832
                 max     5967.044922   11095.305664

**Lectura:** las distancias de Atitlan son varias veces mas grandes que
las de Amatitlan en las dos variables (orilla: mediana 1156 m vs 235 m;
centroide: mediana 4736 m vs 3056 m), simplemente porque Atitlan es un
lago mucho mas grande. Esto confirma que estas variables tienen escalas
muy distintas segun el lago y es una razon mas para que el one-hot de
`lago` acompane a estas distancias: sin esa columna el modelo no podria
distinguir una celda cerca de la orilla en Atitlan (donde 500 m ya es
relativamente cerca) de una en Amatitlan (donde 500 m ya es zona media).


### 3.3.d Vecindad espacial de ndwi

`ndwi_vecindad_3x3` promedia el `ndwi` de la vecindad de 3x3 celdas de
50 m alrededor de cada celda, en la misma fecha. La idea es darle al
modelo algo de contexto local en vez de tratar cada celda como
completamente independiente de sus vecinas: una celda con `ndwi` bajo
rodeada de vecinas tambien bajas es una senal mas fuerte de zona afectada
que una celda aislada. Solo se promedian los vecinos que efectivamente
existen en el conjunto de datos (una celda descartada en el ejercicio 1
por falta de pixeles validos simplemente no aporta al promedio de sus
vecinas; no se inventa un valor para ella).


In [7]:
from src.features import agregar_ndwi_vecindad

enriquecida = agregar_ndwi_vecindad(enriquecida)
display(enriquecida[['ndwi', 'ndwi_vecindad_3x3']].describe())


,ndwi,ndwi_vecindad_3x3
count,492677.000000,492677.000000
mean,0.427291,0.427875
std,0.183100,0.178966
min,-0.607911,-0.385757
25%,0.289169,0.292440
50%,0.412892,0.414095
75%,0.540163,0.540168
max,1.000000,1.000000


**Lectura:** el promedio global casi no cambia (0.427 vs 0.428), pero
la desviacion estandar baja un poco (0.183 vs 0.179) y los extremos se
suavizan de forma visible: el minimo pasa de -0.608 a -0.386. Es el
comportamiento esperado de un promedio de vecindad, que actua como un
suavizado espacial y le da al modelo una version de `ndwi` menos
sensible al ruido de una sola celda.


## 4. Matriz final de predictores

Se combinan todas las columnas numericas anteriores con el one-hot de
`lago` y `estacion`, se verifica que ninguna variable prohibida se haya
colado (`verificar_anti_fuga`), se descartan las pocas filas donde
`ratio_B03_B08` quedo indefinido, y se escribe la matriz junto con su
diccionario.


In [8]:
from src.features import (
    columnas_predictoras,
    construir_diccionario,
    construir_matriz_predictores,
    escribir_diccionario,
    escribir_features,
)

matriz = construir_matriz_predictores(tabla_resp)
ruta_features = escribir_features(matriz)
filas_dic = construir_diccionario(matriz.columns)
ruta_dic = escribir_diccionario(filas_dic)

print(f"Matriz de predictores: {len(matriz)} filas, {len(columnas_predictoras(matriz))} predictores en {ruta_features}")
print(f"Diccionario escrito en {ruta_dic}")
display(matriz.head())


Matriz de predictores: 492663 filas, 17 predictores en D:\Tareas\Data Science\Laboratorio 4 Parte 2\data\processed\ml\features_ml.parquet
Diccionario escrito en D:\Tareas\Data Science\Laboratorio 4 Parte 2\results\tables\diccionario_predictores.csv


,B03,B08,ndwi,x_utm,y_utm,mes,dia_anio_sin,dia_anio_cos,frac_valida,ratio_B03_B08,dist_orilla_m,dist_centroide_m,ndwi_vecindad_3x3,lago_amatitlan,lago_atitlan,estacion_lluviosa,estacion_seca,cyano_alta
0,0.026318,0.019153,0.158667,757645.0,1603375.0,1,0.463258,0.886223,0.68,1.374079,31.300333,5949.483398,0.177437,1,0,0,1,0
1,0.026971,0.020192,0.147370,757695.0,1603375.0,1,0.463258,0.886223,0.96,1.335741,39.232059,5910.458008,0.178495,1,0,0,1,0
2,0.025148,0.018965,0.142702,757745.0,1603375.0,1,0.463258,0.886223,0.92,1.325997,40.331421,5871.599121,0.172630,1,0,0,1,0
3,0.025006,0.019106,0.132742,757795.0,1603375.0,1,0.463258,0.886223,0.64,1.308799,22.230629,5832.909668,0.166390,1,0,0,1,0
4,0.030995,0.021941,0.170967,757595.0,1603325.0,1,0.463258,0.886223,0.88,1.412679,52.396393,5957.851074,0.186989,1,0,0,1,0


In [9]:
diferencia = len(tabla_resp) - len(matriz)
print(f"Dataset base: {len(tabla_resp)} filas. Matriz final: {len(matriz)} filas. "
      f"Diferencia: {diferencia} ({100 * diferencia / len(tabla_resp):.3f}%), "
      "descartadas por ratio_B03_B08 indefinido."
)


Dataset base: 492677 filas. Matriz final: 492663 filas. Diferencia: 14 (0.003%), descartadas por ratio_B03_B08 indefinido.


### Chequeo de alarma: correlacion de Spearman contra cianobacteria_ugl

Cada predictor se compara, por curiosidad y como control de calidad
adicional, contra `cianobacteria_ugl` (la variable continua de la que
sale `cyano_alta`, que nunca entra a la matriz). Un `|rho|` de Spearman
por encima de 0.95 se reportaria como alarma de posible fuga oculta que
`VARIABLES_EXCLUIDAS_RESPUESTA` no haya anticipado, pero no se elimina
ninguna variable automaticamente: el enunciado del laboratorio pide
juicio humano para esa decision, no un corte mecanico.


In [10]:
from scipy.stats import spearmanr

from src.features import UMBRAL_ALARMA_SPEARMAN, alarma_correlacion_spearman

referencia = tabla_resp.loc[matriz.index, 'cianobacteria_ugl']

alarmas = alarma_correlacion_spearman(matriz, referencia)
if alarmas:
    display(pd.DataFrame(alarmas))
else:
    print(f"Sin alarmas de correlacion (umbral |rho Spearman| > {UMBRAL_ALARMA_SPEARMAN}).")

filas_rho = []
for columna in columnas_predictoras(matriz):
    valores = matriz[columna]
    if valores.nunique() < 2:
        continue
    rho, _p = spearmanr(valores, referencia)
    filas_rho.append({'variable': columna, 'rho_spearman': round(float(rho), 4)})

tabla_rho = pd.DataFrame(filas_rho).sort_values('rho_spearman', key=lambda s: s.abs(), ascending=False)
display(tabla_rho.reset_index(drop=True))


Sin alarmas de correlacion (umbral |rho Spearman| > 0.95).


,variable,rho_spearman
0,B08,0.6622
1,B03,0.6596
2,frac_valida,0.6075
3,lago_amatitlan,0.5668
4,lago_atitlan,-0.5668
5,ratio_B03_B08,-0.5587
6,ndwi,-0.5537
7,ndwi_vecindad_3x3,-0.5479
8,dist_orilla_m,-0.5240
9,y_utm,-0.4345


**Lectura:** ningun predictor se acerca al umbral de alarma de 0.95.
Los `rho` mas altos son los de `B08` (0.662) y `B03` (0.660), coherente
con la correlacion de Pearson ya vista en el ejercicio 1.5 (0.700 y 0.658
respectivamente): son bandas espectrales cercanas al indice de
cianobacteria y es razonable que carguen buena parte de la senal, sin
que eso implique fuga (a diferencia de B04, esta relacion es indirecta y
fisica, no una dependencia algebraica exacta como la de B04 con el NDCI).

Tambien resalta que `lago_amatitlan`/`lago_atitlan` (0.567 en valor
absoluto) y `frac_valida` (0.608) correlacionan bastante con
`cianobacteria_ugl`. Esto no es un error de ingenieria de
caracteristicas: es reflejo de que casi toda la senal alta de
cianobacteria del dataset viene de Amatitlan (ejercicio 2.4), asi que
cualquier variable ligada a la identidad del lago hereda parte de esa
correlacion. Es una asimetria real de los datos, no algo que haya que
corregir aqui.


## 5. Verificacion final (gate)

Contrato que debe cumplir la matriz de predictores: no puede tener mas
filas que el dataset base, la diferencia por variables derivadas
indefinidas debe ser menor al 1%, ninguna columna prohibida por fuga
puede estar presente, ningun predictor puede tener NaN o infinitos, y el
diccionario debe documentar exactamente las columnas de la matriz (ni
mas ni menos).


In [11]:
from src.features import verificar_features

resumen_final = verificar_features(matriz, filas_diccionario=filas_dic)
print(f"Verificacion correcta: {resumen_final['filas']} filas, {resumen_final['predictores']} predictores.")
print(
    "Filas excluidas por variable derivada indefinida: "
    f"{resumen_final['filas_excluidas_por_variable_derivada_indefinida']}"
)


Verificacion correcta: 492663 filas, 17 predictores.
Filas excluidas por variable derivada indefinida: 14
